# 6) Unit commitment — Exercises

**Goals:** count start-ups and measure the integrality gap, build a unit-commitment MILP with a variable coal minimum output and match PyPSA, compute uplift payments from fixed-commitment prices, and add minimum up and down times — including the plant's history before the day.

The setup cell holds the lecture's day and a `pypsa_day` helper: unit commitment by default, the fixed-commitment LP when you pass a status table, and optional minimum up/down times. Power in MW, money in €.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

T = 24
HOURS = np.arange(T)
LOAD = 1900 + 400 * np.sin(np.pi * (HOURS - 6) / 12).clip(0) + 250 * np.exp(-((HOURS - 19) / 2.0) ** 2)
FREE = 2200 * 0.8 * np.clip(np.sin(np.pi * (HOURS - 6) / 13), 0, None) + 400.0
PLANTS = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150], "p_min_pu": [0.4, 0.3, 0.1],
                       "start_up_cost": [20_000, 8_000, 1_000], "on_before": [1, 0, 0]},
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))


def pypsa_day(plants=PLANTS, status_fixed=None, min_up=None, min_down=None, linearized=False):
    """The lecture's day in PyPSA. Unit commitment by default; with status_fixed, the fixed-commitment LP;
    linearized=True solves PyPSA's LP relaxation of the unit commitment.
    min_up / min_down: dicts {unit: hours}. The history before the day is long (24 h on or off)."""
    n = pypsa.Network()
    n.set_snapshots(HOURS)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD)
    n.add("Generator", "free", bus="HR", p_nom=FREE.max(), p_max_pu=FREE / FREE.max(), marginal_cost=0)
    for gen, r in plants.iterrows():
        if status_fixed is None:
            extra = {}
            if min_up and gen in min_up:
                extra["min_up_time"] = min_up[gen]
            if min_down and gen in min_down:
                extra["min_down_time"] = min_down[gen]
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc, committable=True,
                  p_min_pu=r.p_min_pu, start_up_cost=r.start_up_cost,
                  up_time_before=24 if r.on_before else 0, down_time_before=0 if r.on_before else 24, **extra)
        else:
            on = status_fixed[gen].to_numpy()
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc, p_max_pu=on, p_min_pu=on * r.p_min_pu)
    n.optimize(solver_name="highs", log_to_console=False, progress=False, linearized_unit_commitment=linearized)
    return n

### Warm-ups

**1. Counting start-ups**

Write `count_starts(status, on_before)` for a status table (rows: hours, columns: units, values 0/1) and the status before the day (Series). Return the number of starts per unit. A unit that was off before the day and is on in hour 0 has started.

In [2]:
def count_starts(status: pd.DataFrame, on_before: pd.Series) -> pd.Series:
    """Number of start-ups per unit over the horizon."""
    ### BEGIN SOLUTION
    prev = pd.concat([on_before.to_frame().T, status.iloc[:-1]], ignore_index=True)
    prev.index = status.index
    return (status - prev).clip(lower=0).sum().astype(int)
    ### END SOLUTION

In [3]:
n_uc = pypsa_day()
lecture_status = n_uc.generators_t.status[PLANTS.index].astype(int)
starts = count_starts(lecture_status, PLANTS.on_before)
assert starts.to_dict() == {"coal": 0, "ccgt": 2, "ocgt": 0}, "as in the lecture"
toy = pd.DataFrame({"a": [0, 1, 1, 0, 1], "b": [1, 1, 0, 0, 0]})
assert count_starts(toy, pd.Series({"a": 1, "b": 0})).to_dict() == {"a": 2, "b": 1}, "a restarts at hours 1 and 4"
starts

name
coal    0
ccgt    2
ocgt    0
dtype: int64

**2. The integrality gap**

Write `integrality_gap(milp_cost, relaxed_cost)` returning (absolute gap in €, gap relative to the MILP). Run PyPSA's unit commitment with and without `linearized_unit_commitment=True` — which, for this model, gives exactly the plain LP relaxation of the lecture.

In [4]:
def integrality_gap(milp_cost: float, relaxed_cost: float) -> tuple[float, float]:
    """Absolute (EUR) and relative gap between a MILP and its LP relaxation."""
    ### BEGIN SOLUTION
    gap = milp_cost - relaxed_cost
    return gap, gap / milp_cost
    ### END SOLUTION

In [5]:
n_relaxed = pypsa_day(linearized=True)
gap, rel = integrality_gap(n_uc.objective, n_relaxed.objective)
assert np.isclose(n_uc.objective, 1_880_184.24, atol=0.01) and np.isclose(gap, 34_464, atol=1)
assert np.isclose(rel, 0.0183, atol=1e-4)
print(f"integrality gap {gap:,.0f} EUR ({rel:.2%})")

integrality gap 34,464 EUR (1.83%)


### Core

**3. Your own unit commitment, with a knob**

Write `unit_commitment(coal_p_min_pu)` — the lecture's Linopy MILP with coal's minimum stable output (as a share of its capacity) as a parameter. Return a dict with the cost, the status table (DataFrame, 0/1) and the curtailed solar (MWh). Check it against PyPSA for 0.4 (the lecture) and 0.2 (a retrofitted, more flexible plant). What is the flexibility worth?

In [6]:
def unit_commitment(coal_p_min_pu: float) -> dict:
    """Unit-commitment MILP of the day with a given coal minimum stable output (share of p_nom)."""
    ### BEGIN SOLUTION
    plants = PLANTS.copy()
    plants.loc["coal", "p_min_pu"] = coal_p_min_pu
    m = linopy.Model()
    t, G = pd.RangeIndex(T, name="t"), plants.index
    g = m.add_variables(lower=0, coords=[t, G], name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(FREE, coords=[t]), name="free")
    u = m.add_variables(coords=[t, G], name="u", binary=True)
    s = m.add_variables(lower=0, coords=[t, G], name="start")
    p_max = xr.DataArray(plants.p_nom.to_numpy(), coords=[G])
    p_min = xr.DataArray((plants.p_nom * plants.p_min_pu).to_numpy(), coords=[G])
    m.add_constraints(g - p_max * u <= 0, name="max_when_on")
    m.add_constraints(g - p_min * u >= 0, name="min_when_on")
    m.add_constraints(s - u + u.shift(t=1) >= 0, name="start_up", mask=xr.DataArray(HOURS > 0, coords=[t]))
    m.add_constraints(s.sel(t=0) - u.sel(t=0) >= -xr.DataArray(plants.on_before.to_numpy(), coords=[G]), name="start_up_0")
    m.add_constraints(g.sum("gen") + f == xr.DataArray(LOAD, coords=[t]), name="balance")
    m.add_objective((g * xr.DataArray(plants.mc.to_numpy(), coords=[G])).sum()
                    + (s * xr.DataArray(plants.start_up_cost.to_numpy(), coords=[G])).sum())
    m.solve(solver_name="highs", output_flag=False)
    status = m.variables["u"].solution.to_pandas().round().astype(int)
    status.columns.name = None
    return {"cost": m.objective.value, "status": status,
            "curtailed": float((FREE - m.variables["free"].solution).sum())}
    ### END SOLUTION

In [7]:
FLEXIBLE = PLANTS.copy()
FLEXIBLE.loc["coal", "p_min_pu"] = 0.2
for share, plants in [(0.4, PLANTS), (0.2, FLEXIBLE)]:
    assert np.isclose(unit_commitment(share)["cost"], pypsa_day(plants).objective, rtol=1e-9), share
stiff, flexible = unit_commitment(0.4), unit_commitment(0.2)
assert flexible["cost"] < stiff["cost"] and flexible["curtailed"] < stiff["curtailed"] / 5
print(f"coal minimum 40 %: cost {stiff['cost']:,.0f} EUR, curtailed {stiff['curtailed']:,.0f} MWh")
print(f"coal minimum 20 %: cost {flexible['cost']:,.0f} EUR, curtailed {flexible['curtailed']:,.0f} MWh")
print(f"value of the retrofit: {stiff['cost'] - flexible['cost']:,.0f} EUR on this day")

coal minimum 40 %: cost 1,880,184 EUR, curtailed 727 MWh
coal minimum 20 %: cost 1,840,643 EUR, curtailed 68 MWh
value of the retrofit: 39,541 EUR on this day


**4. Uplift**

Write `uplift(n_fixed, status, plants)` for a solved fixed-commitment network, its status table and the plant data: per unit, the market margin (price − marginal cost) × output, the start-up costs, the profit, and the uplift needed to make it whole. Check the lecture's 16,000 € for the CCGT, then compute it for the flexible-coal schedule.

In [8]:
def uplift(n_fixed, status: pd.DataFrame, plants: pd.DataFrame = PLANTS) -> pd.DataFrame:
    """Market margin, start-up costs, profit and uplift (EUR) per unit at fixed-commitment prices."""
    ### BEGIN SOLUTION
    price = n_fixed.buses_t.marginal_price["HR"].to_numpy()
    rows = {}
    n_starts = count_starts(status, plants.on_before)
    for gen in plants.index:
        margin = float(((price - plants.loc[gen, "mc"]) * n_fixed.generators_t.p[gen].to_numpy()).sum())
        costs = float(n_starts[gen] * plants.loc[gen, "start_up_cost"])
        rows[gen] = {"market_margin": margin, "start_up_costs": costs, "profit": margin - costs,
                     "uplift": max(costs - margin, 0.0)}
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [9]:
lecture = uplift(pypsa_day(status_fixed=lecture_status), lecture_status)
assert np.isclose(lecture.loc["ccgt", "uplift"], 16_000) and np.isclose(lecture.loc["coal", "profit"], 333_000)
flex_status = flexible["status"]
n_flex = pypsa_day(FLEXIBLE, status_fixed=flex_status)
flex = uplift(n_flex, flex_status, FLEXIBLE)
print("lecture schedule (coal minimum 40 %):")
print(lecture.round(0))
print("\nflexible coal (minimum 20 %):")
print(flex.round(0))
print("\nprices, flexible coal:", (n_flex.buses_t.marginal_price["HR"].round(1) + 0.0).tolist())

lecture schedule (coal minimum 40 %):
      market_margin  start_up_costs    profit   uplift
coal       333000.0             0.0  333000.0      0.0
ccgt            0.0         16000.0  -16000.0  16000.0
ocgt            0.0             0.0       0.0      0.0

flexible coal (minimum 20 %):
      market_margin  start_up_costs    profit   uplift
coal       419400.0             0.0  419400.0      0.0
ccgt            0.0         16000.0  -16000.0  16000.0
ocgt            0.0             0.0       0.0      0.0

prices, flexible coal: [95.0, 95.0, 95.0, 95.0, 95.0, 95.0, 95.0, 95.0, 60.0, 60.0, 60.0, 60.0, 0.0, 0.0, 60.0, 60.0, 60.0, 60.0, 95.0, 95.0, 95.0, 95.0, 95.0, 95.0]


**5. Interpretation**

Making coal more flexible changes the curtailment, the total cost and coal's own margin. Explain each change from the prices. Why does the CCGT still need an uplift — and why is a start-up cost a *non-convex* cost that breaks the link between marginal prices and cost recovery that held in every LP so far?

<!-- BEGIN SOLUTION -->
With a 20 % minimum, coal can go down to 180 MW instead of 360 MW at midday, so free solar is pushed out in far fewer hours: the price is 0 €/MWh only from 12:00 to 13:59 instead of from 11:00 to 15:59, curtailment falls from 727 to 68 MWh, and the day is 39,541 € cheaper — the value of the retrofit on this day. Coal's own margin rises from 333,000 to 419,400 €, because it now spends fewer hours producing at a price of zero, and produces less in those hours; every MWh it sells at 0 €/MWh costs it 60 € in fuel. The CCGT still needs its 16,000 € uplift: whenever it runs it is the marginal plant, so the price equals its own marginal cost and it never earns anything towards its two starts — flexibility elsewhere does not change that.

A start-up cost is paid once when a unit switches on, however much it then produces: the cost of producing a small amount is not small, and cost as a function of output jumps at zero — it is not convex. In an LP the KKT conditions guarantee that the market price supports the optimal schedule: every unit earns at least its marginal cost on every MWh it produces and is content with its own dispatch. With non-convex costs no set of hourly prices may support the optimal schedule — at the fixed-commitment prices a plant that belongs to the cheapest schedule (the CCGT) recovers only its marginal cost, never its start-ups. That is why markets need uplift payments, or offers such as block orders that carry the non-convex costs themselves.
<!-- END SOLUTION -->

### Challenge

**6. Minimum up and down times**

Real units must stay on for a minimum time once started, and off for a minimum time once stopped. With a start variable s and a stop variable d (u[t] − u[t−1] = s[t] − d[t]) they are:

- **min up time UT**: Σ_{k = t−UT+1}^{t} s[k] ≤ u[t]
- **min down time DT**: Σ_{k = t−DT+1}^{t} d[k] ≤ 1 − u[t]

Write `uc_min_times(min_up, min_down)` with these constraints (dicts {unit: hours}; the history before the day is long, so it imposes nothing) and match PyPSA's `min_up_time` / `min_down_time` for the three cases below. Which cases change the schedule?

In [10]:
def uc_min_times(min_up: dict, min_down: dict) -> dict:
    """Unit commitment with minimum up/down times. Returns cost (EUR) and status (DataFrame)."""
    ### BEGIN SOLUTION
    m = linopy.Model()
    t, G = pd.RangeIndex(T, name="t"), PLANTS.index
    g = m.add_variables(lower=0, coords=[t, G], name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(FREE, coords=[t]), name="free")
    u = m.add_variables(coords=[t, G], name="u", binary=True)
    s = m.add_variables(lower=0, coords=[t, G], name="start")
    d = m.add_variables(lower=0, coords=[t, G], name="stop")
    p_max = xr.DataArray(PLANTS.p_nom.to_numpy(), coords=[G])
    p_min = xr.DataArray((PLANTS.p_nom * PLANTS.p_min_pu).to_numpy(), coords=[G])
    m.add_constraints(g - p_max * u <= 0, name="max_when_on")
    m.add_constraints(g - p_min * u >= 0, name="min_when_on")
    m.add_constraints(u - u.shift(t=1) - s + d == 0, name="switching", mask=xr.DataArray(HOURS > 0, coords=[t]))
    m.add_constraints(u.sel(t=0) - s.sel(t=0) + d.sel(t=0) == xr.DataArray(PLANTS.on_before.to_numpy(), coords=[G]),
                      name="switching_0")
    for gen in G:
        up, down = min_up.get(gen, 1), min_down.get(gen, 1)
        for hour in range(T):
            if up > 1:
                window = range(max(0, hour - up + 1), hour + 1)
                m.add_constraints(sum(s.sel(t=k, gen=gen) for k in window) - u.sel(t=hour, gen=gen) <= 0,
                                  name=f"min_up_{gen}_{hour}")
            if down > 1:
                window = range(max(0, hour - down + 1), hour + 1)
                m.add_constraints(sum(d.sel(t=k, gen=gen) for k in window) + u.sel(t=hour, gen=gen) <= 1,
                                  name=f"min_down_{gen}_{hour}")
    m.add_constraints(g.sum("gen") + f == xr.DataArray(LOAD, coords=[t]), name="balance")
    m.add_objective((g * xr.DataArray(PLANTS.mc.to_numpy(), coords=[G])).sum()
                    + (s * xr.DataArray(PLANTS.start_up_cost.to_numpy(), coords=[G])).sum())
    m.solve(solver_name="highs", output_flag=False)
    return {"cost": m.objective.value, "status": m.variables["u"].solution.to_pandas().round().astype(int)}
    ### END SOLUTION

In [11]:
cases = {"CCGT 6 h up and down": ({"ccgt": 6}, {"ccgt": 6}),
         "CCGT 12 h down": ({}, {"ccgt": 12}),
         "CCGT 12 h up": ({"ccgt": 12}, {})}
rows = {}
for name, (up, down) in cases.items():
    mine = uc_min_times(up, down)
    ref = pypsa_day(min_up=up, min_down=down)
    assert np.isclose(mine["cost"], ref.objective, rtol=1e-9), name
    rows[name] = {"cost": mine["cost"], "extra_cost": mine["cost"] - n_uc.objective,
                  "ccgt_hours_on": int(mine["status"]["ccgt"].sum())}
pd.DataFrame(rows).T.round(0)

                           cost  extra_cost  ccgt_hours_on
CCGT 6 h up and down  1880184.0         0.0           14.0
CCGT 12 h down        1918255.0     38071.0           12.0
CCGT 12 h up          1937918.0     57734.0           18.0

**7. Interpretation**

The 6-hour limits change nothing; the 12-hour limits cost tens of thousands of euros. Why? And PyPSA needs `up_time_before` / `down_time_before`: what would go wrong if you told it the CCGT had been off for only one hour before the day, with a 6-hour minimum down time — and why does this matter when a model is run day after day?

<!-- BEGIN SOLUTION -->
Minimum times only cost something when they forbid the schedule the model wanted anyway. The lecture's CCGT runs 8 hours at night, rests 10 hours and runs 6 in the evening — every block is at least 6 hours long, so 6-hour limits are not binding. A 12-hour minimum down time forbids a 10-hour rest: the CCGT has to stop earlier (and the expensive OCGT or coal at its limit must fill in) so that it can be back for the evening peak, which costs about 38,000 €. A 12-hour minimum up time forbids stopping after 8 hours: the CCGT must keep running into the sunny hours, displacing cheaper energy, which costs even more. The history matters because minimum times look back in time: a unit that has been off for only one hour with a 6-hour minimum down time must stay off for five more hours. Told that, PyPSA keeps the CCGT off in the first night hours — and since coal, free energy and the OCGT cannot cover the night demand, the problem becomes infeasible. In real operation the history is not a free choice: when a model is run day after day (a rolling horizon, lesson 9), each day must start from the status and up/down times at which the previous day ended, or the schedules are not physically consistent.
<!-- END SOLUTION -->